In [5]:
# baseCode 

from google import genai
from google.genai import types
import os

client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
model = "gemini-2.5-flash-lite"
def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "parts": [{"text": text}]
    })

def add_assistant_message(messages, text):
    messages.append({
        "role": "model",
        "parts": [{"text": text}]
    })

def chatwithstreamfunction(messages, system_prompt=None, temperature=1.0 , stop_sequences=[]):
    
    config = {
        "max_output_tokens": 100000,
        "temperature": temperature,
        "stop_sequences" : stop_sequences
    }

    if system_prompt:
        config["system_instruction"] = system_prompt

   
    stream = client.models.generate_content_stream(
        model=model,
        contents=messages,
        config=types.GenerateContentConfig(**config)
    )
    full_text = ""
    for chunk in stream:
        if chunk.text:
            # Send chunk to client (like Claude text_stream)
            print(chunk.text, end="", flush=True)
            full_text += chunk.text

    # Equivalent of get_final_message()
    return full_text 

# def chatwithgeneratefunction(messages , system_prompt=None, temperature=1.0 , stop_sequences=[]):
#     config = {
#         "max_output_tokens": 100000,
#         "temperature": temperature,
#         "stop_sequences" : stop_sequences
#     }

#     if system_prompt:
#         config["system_instruction"] = system_prompt
    
#     response = client.models.generate_content(
#         model=model,
#         contents=messages,
#         config=types.GenerateContentConfig(**config)
#     )
#     return response.text[0]
     

In [6]:
messages=[]
import json
add_user_message(messages , "generate  a very short event bridge rule as json")
add_assistant_message(messages , "```json")
reply = chatwithstreamfunction(messages, stop_sequences=["```"])
print(reply)
print(json.loads(reply.strip()))


{
  "Name": "MySimpleRule",
  "EventBusName": "default",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Description": "Rule to trigger on EC2 instance state change to running"
}

{
  "Name": "MySimpleRule",
  "EventBusName": "default",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Description": "Rule to trigger on EC2 instance state change to running"
}

{'Name': 'MySimpleRule', 'EventBusName': 'default', 'EventPattern': {'source': ['aws.ec2'], 'detail-type': ['EC2 Instance State-change Notification'], 'detail': {'state': ['running']}}, 'State': 'ENABLED', 'Description': 'Rule to trigger on EC2 instance state change to running'}


Exercise !
use message prefilling and stop sequences only to get three different commands in single response 
there shouldn't be any comments or explanation
Hint: message prefilling isn't limited to just characters like ```

In [14]:
messages=[]

prompt = "generate three different sample aws cli command. each should be very short"

add_user_message(messages, prompt)
add_assistant_message(messages,"Here are all three command in a single block without any comments :\n ```bash")
text = chatwithstreamfunction(messages , stop_sequences=["```"])
text.strip()


 aws s3 ls
 aws ec2 describe-instances --query 'Reservations[*].Instances[*].InstanceId'
 aws iam list-users
 

"aws s3 ls\n aws ec2 describe-instances --query 'Reservations[*].Instances[*].InstanceId'\n aws iam list-users"